In [0]:
from pyspark.sql import Window

# ============================================================
# Silver quality: rules derived from the contract and the mapping (nobody writes them by hand).
#   row rules (-> quarantine): not_null, conversion, accepted_values, range, reference_exists
#   batch rules (-> warn):     duplicate_pk (latest bronze version kept), completeness
# A batch whose quarantined share exceeds max_quarantine_ratio gets status 'fail' (nothing is written).
# Everything happens in memory; writing (MERGE, quarantine table, logs) is done by the silver framework.
# Requires: common/_utils_config, common/_utils_reference_data, silver/utils/_utils_silver_contracts,
#           silver/utils/_utils_silver_mappings, silver/utils/_utils_silver_parser
# Cells: 1 actions/reference tables · 2 evaluation · 3 report and dry run
# ============================================================

quality_rule_actions = {
    "not_null": "quarantine",
    "conversion": "quarantine",
    "accepted_values": "quarantine",
    "range": "quarantine",
    "reference_exists": "quarantine",
    "duplicate_pk": "warn",
    "completeness": "warn",
}
completeness_sample_size = 10


def with_columns(df, columns):
    """withColumns that accepts an empty mapping (Spark Connect asserts on empty withColumns)."""
    return df.withColumns(columns) if columns else df


def reference_source_rows(contract, contracts):
    """Rows (dicts) of a reference entity: its csv, or the data_source rows built from the source specs."""
    if contract.get("reference_source"):
        return build_data_source_rows(contracts)
    return read_reference_csv(contract["reference_file"])


def build_reference_df(contract, contracts, silver_run_id="dry_run"):
    """Reference table as a DataFrame (PK, FKDataSource, typed contract columns, lineage), built from its origin."""
    use_utc_session()
    columns = contract["columns"]
    rows = [tuple(None if row.get(c["name"]) in (None, "") else str(row.get(c["name"])) for c in columns)
            for row in reference_source_rows(contract, contracts)]
    df = spark.createDataFrame(rows, ", ".join(f"`{c['name']}` STRING" for c in columns))
    df = with_columns(df, {c["name"]: F.expr(f"try_cast(`{c['name']}` AS {c['type']})")
                           for c in columns if c["type"] != "string"})

    table = build_silver_tables(contract)[0]
    types = {c["name"]: c["type"] for c in columns}
    keys = {table["primary_key"]["name"]: key_hash_sql([(c, types[c]) for c in table["primary_key"]["columns"]])}
    for fk in table["foreign_keys"]:
        keys[fk["name"]] = (f"md5({sql_string(fk['constant'])})" if fk["constant"]
                            else key_hash_sql([(c, types[c]) for c in fk["columns"]]))
    df = with_columns(df, {name: F.expr(sql) for name, sql in keys.items()})

    origin = (f"{reference_data_folder}/{contract['reference_file']}" if contract.get("reference_file")
              else f"{source_specs_folder}/")
    df = df.withColumns({
        "bronze_table": F.lit(None).cast("string"),
        "source_file": F.lit(origin),
        "chunk_start": F.lit(None).cast("date"),
        "chunk_end": F.lit(None).cast("date"),
        "bronze_run_id": F.lit(None).cast("string"),
        "silver_run_id": F.lit(silver_run_id),
        "processed_at": F.current_timestamp(),
    })
    return df.select(*[name for name, _ in table["columns"]])

In [0]:
# ------------------------------------------------------------
# Quality evaluation: parsed batch -> valid rows, quarantined rows, metrics, status
# ------------------------------------------------------------
def row_rule_conditions(contract, mapping):
    """[(rule, column, failure condition)] evaluated row by row (except reference_exists, added with joins)."""
    rules = mapping["columns"]
    keys = set(contract["business_keys"])
    conditions = []
    for column in contract["columns"]:
        name = column["name"]
        value = F.col(name)
        if name in keys or column.get("nullable") is False:
            conditions.append(("not_null", name, value.isNull()))
        if "from" in rules[name]:
            conditions.append(("conversion", name, F.col(f"{raw_prefix}{name}").isNotNull() & value.isNull()))
        if column.get("accepted_values"):
            conditions.append(("accepted_values", name, value.isNotNull() & ~value.isin(column["accepted_values"])))
        if column.get("range"):
            value_range = column["range"]
            outside = F.lit(False)
            if "min" in value_range:
                outside = outside | (value < value_range["min"])
            if "max" in value_range:
                outside = outside | (value > value_range["max"])
            conditions.append(("range", name, value.isNotNull() & outside))
    return conditions


def evaluate_completeness(valid_df, contract):
    """Per group and period, without the first and last period (partial at batch edges):
    expected_rows -> number of rows; resolution_column -> minutes covered (sum of resolutions = 1440 a day),
    so days that change resolution (e.g. hourly -> quarter-hourly) are complete when fully covered."""
    completeness = contract.get("completeness")
    if not completeness:
        return None, []
    time_column = completeness["time_column"]
    group_by = list(completeness.get("group_by", []))
    resolution = completeness.get("resolution_column")
    period_start = F.to_date(time_column) if completeness["period"] == "day" else F.trunc(time_column, "month")

    grouped = valid_df.groupBy(*group_by, period_start.alias("period_start"))
    if resolution:
        counts = (grouped.agg(F.sum(resolution).alias("count"))
                  .withColumns({"expected": F.lit(1440), "measure": F.lit("minutes")}))
    else:
        counts = (grouped.agg(F.count(F.lit(1)).alias("count"))
                  .withColumns({"expected": F.lit(completeness["expected_rows"]), "measure": F.lit("rows")}))

    periods = [r[0] for r in counts.select("period_start").distinct().orderBy("period_start").collect()]
    checked = counts.filter(F.col("period_start").isin(periods[1:-1]))
    issues_df = checked.filter(F.col("count") != F.col("expected"))

    metric = {"rule": "completeness", "column": time_column, "action": quality_rule_actions["completeness"],
              "evaluated": checked.count(), "failed": issues_df.count()}
    issues = [r.asDict() for r in issues_df.orderBy("period_start").limit(completeness_sample_size).collect()]
    return metric, issues


def evaluate_quality(parsed_df, mapping, contracts):
    """Returns a dict: valid (DataFrame), quarantine (DataFrame), metrics, counts, ratio, status, issues."""
    contract = contracts[mapping["entity"]]
    table = build_silver_tables(contract)[0]
    pk = table["primary_key"]["name"]
    silver_columns = [name for name, _ in table["columns"]]
    raw_columns = [c for c in parsed_df.columns if c.startswith(raw_prefix)]

    df = parsed_df
    conditions = row_rule_conditions(contract, mapping)
    for fk in table["foreign_keys"]:
        reference_column = f"__ref_{fk['name']}"
        reference = (build_reference_df(contracts[fk["references_table"]], contracts)
                     .select(F.col(fk["references_column"]).alias(reference_column)).distinct())
        df = df.join(F.broadcast(reference), F.col(fk["name"]) == F.col(reference_column), "left")
        conditions.append(("reference_exists", fk["name"],
                           F.col(fk["name"]).isNotNull() & F.col(reference_column).isNull()))

    flags = [(f"__fail_{i}", rule, column) for i, (rule, column, _) in enumerate(conditions)]
    df = df.withColumns({flag: F.coalesce(condition, F.lit(False))
                         for (flag, _, _), (_, _, condition) in zip(flags, conditions)})
    df = df.withColumn("failed_rules", F.filter(
        F.array(*[F.when(F.col(flag), F.lit(f"{rule}:{column}")) for flag, rule, column in flags]),
        lambda item: item.isNotNull()))

    stats = df.agg(F.count(F.lit(1)).alias("rows"),
                   *[F.sum(F.col(flag).cast("int")).alias(flag) for flag, _, _ in flags]).first().asDict()
    total = stats["rows"]
    metrics = [{"rule": rule, "column": column, "action": quality_rule_actions[rule],
                "evaluated": total, "failed": stats[flag] or 0} for flag, rule, column in flags]

    has_failed = F.size("failed_rules") > 0
    raw_values = (F.create_map(*[part for c in raw_columns for part in (F.lit(c[len(raw_prefix):]), F.col(c))])
                  if raw_columns else F.create_map().cast("map<string,string>"))
    quarantine = df.filter(has_failed).select(*silver_columns, "failed_rules", raw_values.alias("raw_values"),
                                              F.current_timestamp().alias("quarantined_at"))

    passed = df.filter(~has_failed)
    latest_first = Window.partitionBy(pk).orderBy(F.desc("chunk_start"), F.desc("source_file"))
    valid = (passed.withColumn("__version", F.row_number().over(latest_first))
                   .filter(F.col("__version") == 1).select(*silver_columns))
    passed_count, valid_count = passed.count(), valid.count()
    metrics.append({"rule": "duplicate_pk", "column": pk, "action": quality_rule_actions["duplicate_pk"],
                    "evaluated": passed_count, "failed": passed_count - valid_count})

    completeness_metric, completeness_issues = evaluate_completeness(valid, contract)
    if completeness_metric:
        metrics.append(completeness_metric)

    quarantined = total - passed_count
    ratio = quarantined / total if total else 0.0
    if total == 0:
        status = "empty"
    elif ratio > max_quarantine_ratio:
        status = "fail"
    elif quarantined or any(m["failed"] for m in metrics):
        status = "warning"
    else:
        status = "ok"

    return {
        "mapping": mapping["mapping"],
        "status": status,
        "rows": total,
        "valid_rows": valid_count,
        "quarantined_rows": quarantined,
        "quarantine_ratio": ratio,
        "metrics": metrics,
        "completeness_issues": completeness_issues,
        "valid": valid,
        "quarantine": quarantine,
    }

In [0]:
# ------------------------------------------------------------
# Quality report and dry run with quality
# ------------------------------------------------------------
status_icons = {"ok": "✅", "warning": "⚠️", "fail": "⛔", "empty": "∅"}


def print_quality_report(result):
    print(f"{status_icons[result['status']]} Quality · {result['mapping']} · status {result['status']}")
    print(f"   rows {result['rows']} · valid {result['valid_rows']} · quarantined {result['quarantined_rows']} "
          f"({result['quarantine_ratio']:.1%}, limit {max_quarantine_ratio:.0%})")
    print("   rule              column                       action      evaluated   failed")
    for m in result["metrics"]:
        flag = " ⚠️" if m["failed"] else ""
        print(f"   {m['rule']:<17} {m['column']:<28} {m['action']:<11} {m['evaluated']:>9}   {m['failed']}{flag}")
    for issue in result["completeness_issues"]:
        print(f"   ⚠️ incomplete: {issue}")
    if result["quarantined_rows"]:
        display(result["quarantine"].limit(10))


def dry_run_with_quality(mapping, contracts, files=dry_run_files):
    """Parse the latest bronze files and evaluate quality. Returns (parsed, parse report, quality result)."""
    parsed, report = dry_run_mapping(mapping, contracts, files)
    return parsed, report, evaluate_quality(parsed, mapping, contracts)